# Detailed variant statistics for ExCALIBR
Here we get detailed variant statistics for variant groups of interest

In [ ]:
# Make sure our package is installed (run once)
#!pip install -e /home/dataproc/repos/igvf-cvfg-biobank

Notes on running in AoU's DataProc: as of 2026-09-25, trying to use pandas with the `fastparquet` engine throws an error. Manually forcing an update of relevant packages resolves it:

In [ ]:
# Run once in dataproc
#!pip install --upgrade numpy numba fastparquet pandas statsmodels

In the standard environment, `fastparquet` is not installed, but we use it to append to a parquet file:

In [ ]:
# Run once in a standard env
#!pip install fastparquet

In [ ]:
import os
import pandas as pd
from pathlib import Path
from functools import cache
import hail as hl
from cvfgaou import notation, hailtools
from tqdm.notebook import tqdm

In [ ]:
# In standard env
#wgs_mt_path = "/home/jupyter/workspace/All of Us Controlled Tier v8/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/splitMT/hail.mt"

In [ ]:
# In Dataproc
wgs_mt_path = 'gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/splitMT/hail.mt'

In [ ]:
# Constants (mostly paths to things)
bucket_mount_path = Path.home() / 'workspace' / 'scalable-vus-validation'

# Inputs
variant_table_path = bucket_mount_path / 'variant_tables/integrated_variant_effect_biobank_input_data_excalibr_assays_oddspath_models.tsv.gz'
gene_cohorts_path = bucket_mount_path / 'gene_cohorts'

google_project = os.getenv('GOOGLE_PROJECT')

excalibr_variant_counts_path = bucket_mount_path / 'variant_groups/excalibr-variant-counts.parquet'

# Index column labels
index_cols=['Dataset/Predictor', 'Gene', 'Grouping/Calibration', 'Consequence', 'Group/Range']

In [ ]:
# Load dataframe
variants_df = pd.read_table(variant_table_path, dtype=str)

In [ ]:
variants_df

Now establish the sets of variants to test.

First we inspect the columns that will define variant sets:

In [ ]:
variants_df[['condensed_consequence']].value_counts(dropna=False)

In [ ]:
variants_df['ExCALIBR_points'].value_counts(dropna=False)

In [ ]:
# Limit to genes with case/control cohorts established
genes_with_cohorts = {
    gene for gene in variants_df['Gene'].unique()
    if Path(gene_cohorts_path, f'{gene} cohort.parquet').exists()
}

In [ ]:
genes_with_cohorts

Function for retrieving cohort hail table efficiently

In [ ]:
@cache
def get_cohort_ht(gene):
    cohort_df = pd.read_parquet(Path(gene_cohorts_path, f'{gene} cohort.parquet'))
    cohort_df['s'] = cohort_df.index.to_series().astype(str)
    return hl.Table.from_pandas(
        cohort_df,
        key='s'
    ).cache()

## ExCALIBR

In [ ]:
variant_groups = {
    (dataset, gene, 'ExCALIBR', consequence, classification):
        df[classification_selection & consequence_selection]
    for (dataset, gene), df in variants_df.groupby(['Dataset', 'Gene'])
    for classification, classification_selection in [
        (f'{notation.GEQ_CHAR} {points:+d}', pd.to_numeric(df['ExCALIBR_points']) >= points)
        for points in range(1,9)
    ] + [
        (f'{notation.LEQ_CHAR} {-points:+d}', pd.to_numeric(df['ExCALIBR_points']) <= -points)
        for points in range(1,9)
    ]
    for consequence, consequence_selection in (
        ('All', True),
        ('Missense', df['condensed_consequence'] == 'missense_variant')
    )
    if gene in genes_with_cohorts
}

Next, get the counts for each variant group

In [ ]:
# First, make sure the position column is an integer and
# make sure that the chromosomes are strings with the "chr" prefix

variant_groups = {
    group_label: df.assign(
        Chromosome = 'chr' + df['Chrom'],
        pos = pd.to_numeric(df['hg38_start'], downcast='integer')
    )
    for group_label, df in variant_groups.items()
}

In [ ]:
# Check for missing NT coordinates
for label, df in variant_groups.items():
    if df[['Chromosome', 'pos', 'ref_allele', 'alt_allele']].isna().any(axis=None):
        display(label)
        display(df)

In [ ]:
hl.init(
    #default_reference='GRCh38', deprecated
    gcs_requester_pays_configuration=google_project
)

Run the counter functions

In [ ]:
# WGS table

wgs_mt = hl.read_matrix_table(wgs_mt_path)

In [ ]:
# Attempt to check if we are adding to an existing file
excluded_groups = set()
if excalibr_variant_counts_path.exists():
    results_df = pd.read_parquet(excalibr_variant_counts_path)
    excluded_groups = set(
        results_df.
        index.
        to_frame()[index_cols].
        drop_duplicates().
        itertuples(index=False, name=None)
    )
    display(f'Existing file found with {len(excluded_groups)} groups.')
else:
    results_df = None

for variant_group, df in tqdm(variant_groups.items()):
    if variant_group in excluded_groups:
        continue

    result = {
        'Variants in Group': df[['Chromosome', 'pos', 'ref_allele', 'alt_allele']].drop_duplicates().shape[0]
    }
    
    if not df.empty:
        result |= hailtools.get_variant_counts(
            hailtools.get_filtered_mt(
                df[['Chromosome', 'pos', 'ref_allele', 'alt_allele']], wgs_mt,
                contig_col='Chromosome',
                pos_col='pos',
                ref_col='ref_allele',
                alt_col='alt_allele',
                reference_genome='GRCh38'
            ),
            get_cohort_ht(variant_group[1]) # Gene is the 2nd field
        )
    else:
        result |= {
            'Variants in WGS MT': None,
            'Variants in cohort': 0,
            'Case only variant count': 0,
            'Control only variant count': 0,
            'Overlap variant count': 0
        }
    
    result_df = pd.DataFrame.from_dict({variant_group: result}, orient='index')
    result_df.index.set_names(index_cols, inplace=True)

    if results_df is None:
        result_df = results_df
    else:
        results_df = pd.concat([results_df, result_df])
        
    results_df.to_parquet(excalibr_variant_counts_path)